In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv(r"C:\Users\dell\Downloads\Dataset_Fish_Market.csv")
print("="*65)
print("FISH MARKET DATASET - LOGISTIC REGRESSION")
print("="*65)
print(f"\nDataset shape: {df.shape}")
print(f"\nFirst 5 rows:\n{df.head()}")

# Encode Species (target) into numeric
le = LabelEncoder()
df['Species_encoded'] = le.fit_transform(df['Species'])
print(f"\nSpecies encoding mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {i} -> {cls}")

# Features: Weight + Length1 + Length2 + Length3 + Height + Width (6 features)
# Target: Species (encoded)
X = df[['Weight', 'Length1', 'Length2', 'Length3', 'Height', 'Width']]
y = df['Species_encoded']

print(f"\nFeatures (X) shape: {X.shape}")
print(f"Target (y) shape:   {y.shape}")
print(f"Feature columns: {list(X.columns)}")

# Train/Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Testing  samples: {X_test.shape[0]}")

# ── Part a: Logistic Regression Model ──────────────────────────
print("\n" + "="*65)
print("PART a: LOGISTIC REGRESSION MODEL — TRAINING PARAMETERS")
print("="*65)

model = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42)
model.fit(X_train, y_train)

# Print all training parameters
print(f"\nModel Hyperparameters:")
for k, v in model.get_params().items():
    print(f"  {k:25s}: {v}")

print(f"\nNumber of classes   : {model.n_iter_}")
print(f"Iterations to converge: {model.n_iter_[0]}")
print(f"Classes             : {[le.classes_[i] for i in model.classes_]}")

print(f"\nIntercepts (bias terms) — one per class:")
for cls_idx, intercept in zip(model.classes_, model.intercept_):
    print(f"  Class '{le.classes_[cls_idx]}': {intercept:.6f}")

print(f"\nCoefficients — shape {model.coef_.shape}  (one row per class, one col per feature):")
feature_names = list(X.columns)
header = f"  {'Class':<12}" + "".join(f"{f:>12}" for f in feature_names)
print(header)
print("  " + "-" * (12 + 12*len(feature_names)))
for cls_idx, coef_row in zip(model.classes_, model.coef_):
    row = f"  {le.classes_[cls_idx]:<12}" + "".join(f"{c:>12.6f}" for c in coef_row)
    print(row)

# ── Part b: Confusion Matrix & Metrics ─────────────────────────
print("\n" + "="*65)
print("PART b: CONFUSION MATRIX & PERFORMANCE METRICS")
print("="*65)

y_pred = model.predict(X_test)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
class_labels = le.classes_

print("\nConfusion Matrix (rows=Actual, cols=Predicted):")
header = f"  {'':>12}" + "".join(f"{c:>12}" for c in class_labels)
print(header)
print("  " + "-" * (12 + 12*len(class_labels)))
for i, row in enumerate(cm):
    print(f"  {class_labels[i]:>12}" + "".join(f"{v:>12}" for v in row))

# Overall Metrics
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print(f"\n{'─'*40}")
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision : {prec:.4f}  (weighted)")
print(f"  Recall    : {rec:.4f}  (weighted)")
print(f"  F1-Score  : {f1:.4f}  (weighted)")
print(f"{'─'*40}")

# Per-class Metrics
print(f"\nPer-Class Metrics:")
header2 = f"  {'Class':<12}{'Precision':>12}{'Recall':>10}{'F1-Score':>10}"
print(header2)
print("  " + "-" * 44)
p_each = precision_score(y_test, y_pred, average=None, zero_division=0)
r_each = recall_score(y_test, y_pred, average=None, zero_division=0)
f_each = f1_score(y_test, y_pred, average=None, zero_division=0)
for i, cls in enumerate(class_labels):
    print(f"  {cls:<12}{p_each[i]:>12.4f}{r_each[i]:>10.4f}{f_each[i]:>10.4f}")

print("\n" + "="*65)
print("DONE")
print("="*65)

FISH MARKET DATASET - LOGISTIC REGRESSION

Dataset shape: (159, 7)

First 5 rows:
  Species  Weight  Length1  Length2  Length3   Height   Width
0   Bream   242.0     23.2     25.4     30.0  11.5200  4.0200
1   Bream   290.0     24.0     26.3     31.2  12.4800  4.3056
2   Bream   340.0     23.9     26.5     31.1  12.3778  4.6961
3   Bream   363.0     26.3     29.0     33.5  12.7300  4.4555
4   Bream   430.0     26.5     29.0     34.0  12.4440  5.1340

Species encoding mapping:
  0 -> Bream
  1 -> Parkki
  2 -> Perch
  3 -> Pike
  4 -> Roach
  5 -> Smelt
  6 -> Whitefish

Features (X) shape: (159, 6)
Target (y) shape:   (159,)
Feature columns: ['Weight', 'Length1', 'Length2', 'Length3', 'Height', 'Width']

Training samples: 127
Testing  samples: 32

PART a: LOGISTIC REGRESSION MODEL — TRAINING PARAMETERS

Model Hyperparameters:
  C                        : 1.0
  class_weight             : None
  dual                     : False
  fit_intercept            : True
  intercept_scaling       